# SQLite to Snowflake ETL (Colab-Style Workbook)

## Overview
This notebook moves data from a local SQLite `.db` file into Snowflake in a reproducible, step-by-step flow.

### What you will do
- Discover SQLite tables
- Export each table to CSV
- Create matching Snowflake target tables
- Upload files to an internal Snowflake stage
- Load data with `COPY INTO`


## Quick Start

### Before running
1. Set `SQLITE_DB_PATH` to your local `.db` file.
2. Fill Snowflake credentials (or use env vars).
3. Run cells top-to-bottom.

### Required packages
- `pandas`
- `snowflake-connector-python`


## Local PC Notes

### Finding your project folder
If you're running this on your own computer, use your local repo path (not `/workspace/...`).

Example:
- macOS/Linux: `cd ~/Downloads/soccer_codex`
- Windows PowerShell: `cd C:\Users\<you>\Downloads\soccer_codex`


## 0) Environment Setup

### Install dependencies (optional in Colab/local notebooks)


In [ ]:
# Optional: install dependencies if needed
# !pip install -q pandas snowflake-connector-python

## 1) Imports

### Import Python libraries


In [ ]:
from pathlib import Path
import os
import sqlite3
import pandas as pd
import snowflake.connector

## 2) Configuration

### Set file paths, table filters, and Snowflake connection settings


In [ ]:
# ---------- Local paths ----------
SQLITE_DB_PATH = Path("/path/to/your/database.db")  # <-- update
EXPORT_DIR = Path("./sqlite_exports")

# Optional: set to [] to auto-load all tables
TABLES_TO_LOAD = []  # e.g., ["customers", "orders"]

# ---------- Named connection profile ----------
CONNECTIONS = {
    "my_example_connection": {
        "account": "NWLBALI-XLB64593",
        "user": "AWESOMEIBRAHIM2022",
        "authenticator": "externalbrowser",
        "role": "ACCOUNTADMIN",
        "warehouse": "<none selected>",
        "database": "<none selected>",
        "schema": "<none selected>",
    }
}

ACTIVE_CONNECTION = "my_example_connection"
SNOWFLAKE_CONFIG = CONNECTIONS[ACTIVE_CONNECTION].copy()

# Snowflake stage for CSV upload
STAGE_NAME = "SQLITE_INGEST_STAGE"

# Snowflake table naming strategy
TARGET_TABLE_PREFIX = ""  # e.g., "RAW_"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"SQLite DB: {SQLITE_DB_PATH}")
print(f"Export dir: {EXPORT_DIR.resolve()}")
print(f"Using Snowflake connection profile: {ACTIVE_CONNECTION}")


## 3) Helper Functions

### Utility functions for quoting, schema mapping, and DDL generation


In [ ]:
def quote_ident(name: str) -> str:
    return '"' + name.replace('\"', '\\\"') + '"'


def sqlite_decltype_to_snowflake(decltype: str) -> str:
    if decltype is None:
        return "STRING"
    t = decltype.strip().upper()
    if "INT" in t:
        return "NUMBER"
    if any(x in t for x in ["REAL", "FLOA", "DOUB"]):
        return "FLOAT"
    if any(x in t for x in ["NUMERIC", "DECIMAL", "BOOLEAN"]):
        return "NUMBER"
    if any(x in t for x in ["DATE", "TIME"]):
        return "TIMESTAMP_NTZ"
    if "BLOB" in t:
        return "BINARY"
    return "STRING"


def discover_sqlite_tables(conn: sqlite3.Connection):
    q = """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """
    return [row[0] for row in conn.execute(q).fetchall()]


def get_table_columns(conn: sqlite3.Connection, table_name: str):
    # PRAGMA table_info columns: cid, name, type, notnull, dflt_value, pk
    rows = conn.execute(f"PRAGMA table_info({quote_ident(table_name)})").fetchall()
    return [{"name": r[1], "type": r[2]} for r in rows]


def create_snowflake_table_sql(target_table: str, columns: list[dict]) -> str:
    col_defs = []
    for col in columns:
        sf_type = sqlite_decltype_to_snowflake(col.get("type"))
        col_defs.append(f"{quote_ident(col['name'])} {sf_type}")
    cols_sql = ",\n    ".join(col_defs) if col_defs else '"_RAW" STRING'
    return f"CREATE OR REPLACE TABLE {quote_ident(target_table)} (\n    {cols_sql}\n)"

## 4) Extract from SQLite

### Discover tables and export to CSV


In [ ]:
if not SQLITE_DB_PATH.exists():
    raise FileNotFoundError(f"SQLite DB not found: {SQLITE_DB_PATH}")

sqlite_conn = sqlite3.connect(SQLITE_DB_PATH)
all_tables = discover_sqlite_tables(sqlite_conn)
tables = TABLES_TO_LOAD if TABLES_TO_LOAD else all_tables

if not tables:
    raise ValueError("No tables found in SQLite database.")

print(f"Discovered tables ({len(all_tables)}): {all_tables}")
print(f"Tables to load ({len(tables)}): {tables}")

exports = []
for table in tables:
    df = pd.read_sql_query(f"SELECT * FROM {quote_ident(table)}", sqlite_conn)
    csv_path = EXPORT_DIR / f"{table}.csv"
    df.to_csv(csv_path, index=False)
    schema_cols = get_table_columns(sqlite_conn, table)
    exports.append({
        "source_table": table,
        "target_table": f"{TARGET_TABLE_PREFIX}{table}",
        "csv_path": csv_path,
        "columns": schema_cols
    })
    print(f"Exported {table}: {len(df):,} rows -> {csv_path}")

sqlite_conn.close()

## 5) Connect to Snowflake

### Validate config and create internal stage


In [ ]:
required = ["account", "user", "warehouse", "database", "schema"]
missing = [
    k for k in required
    if not SNOWFLAKE_CONFIG.get(k) or str(SNOWFLAKE_CONFIG[k]).startswith("<")
]
if missing:
    raise ValueError(f"Missing Snowflake config values: {missing}")

if not SNOWFLAKE_CONFIG.get("password") and not SNOWFLAKE_CONFIG.get("authenticator"):
    raise ValueError("Provide either 'password' or 'authenticator' in SNOWFLAKE_CONFIG.")

sf_conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
sf_cur = sf_conn.cursor()

sf_cur.execute(f"USE WAREHOUSE {quote_ident(SNOWFLAKE_CONFIG['warehouse'])}")
sf_cur.execute(f"USE DATABASE {quote_ident(SNOWFLAKE_CONFIG['database'])}")
sf_cur.execute(f"USE SCHEMA {quote_ident(SNOWFLAKE_CONFIG['schema'])}")
sf_cur.execute(f"CREATE OR REPLACE STAGE {quote_ident(STAGE_NAME)}")
print(f"Stage ready: {STAGE_NAME}")


## 6) Load into Snowflake

### Create tables, PUT files, and COPY data


In [ ]:
load_summary = []

for item in exports:
    source_table = item['source_table']
    target_table = item['target_table']
    csv_path = item['csv_path']
    columns = item['columns']

    create_sql = create_snowflake_table_sql(target_table, columns)
    sf_cur.execute(create_sql)

    put_sql = f"PUT file://{csv_path.resolve()} @{quote_ident(STAGE_NAME)} AUTO_COMPRESS=TRUE OVERWRITE=TRUE"
    sf_cur.execute(put_sql)

    copy_sql = f"""
    COPY INTO {quote_ident(target_table)}
    FROM @{quote_ident(STAGE_NAME)}/{csv_path.name}.gz
    FILE_FORMAT = (
        TYPE = CSV
        FIELD_OPTIONALLY_ENCLOSED_BY = '\"'
        SKIP_HEADER = 1
        EMPTY_FIELD_AS_NULL = TRUE
    )
    ON_ERROR = 'CONTINUE'
    """
    sf_cur.execute(copy_sql)

    count_sql = f"SELECT COUNT(*) FROM {quote_ident(target_table)}"
    loaded_rows = sf_cur.execute(count_sql).fetchone()[0]

    load_summary.append({
        "source_table": source_table,
        "target_table": target_table,
        "rows_loaded": loaded_rows,
        "csv": str(csv_path),
    })

    print(f"Loaded {source_table} -> {target_table}: {loaded_rows:,} rows")

## 7) Validate Results

### Review loaded row counts


In [ ]:
pd.DataFrame(load_summary).sort_values("target_table")

## 8) Cleanup

### Close Snowflake connection


In [ ]:
sf_cur.close()
sf_conn.close()
print("Snowflake connection closed.")